# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis**: One row = one `content_hash_id`, for one `client_hash_id`, on one report_date — a content × day grain in `fact_content_daily_performance`, joined to `dim_content` (static content attributes) on `content_hash_id`.

**Table(s)**: `fact_content_daily_performance` (daily metrics: impressions, clicks, sessions, AI-referral counts) joined to `dim_content` (static: `content_type`, `word_count`, `main_intent`, `provider_used`, etc.) via `content_hash_id`.

**Time window**: September 2025 — a full calendar month, chosen as a genuine mid-panel month. The full panel spans 18 months (2025-01 through 2026-06, confirmed via list_repo_files against the month-partitioned parquet structure, not by streaming), so September sits near the true midpoint — clear of both the panel's start (no prior history to build features from) and its end (no future window left to construct a label from). Loaded directly via DuckDB against the month=2025-09 partition, giving 845,813 rows — the full month, no sampling needed.

**What I'd predict/rank**: No pre-built label exists in this table. The proxy label is constructed from `gsc_impressions`: sum it over a prior sub-window vs. a later sub-window within September, compute % change, then bucket into up/down/stable.

Deliberately excluded: All hash IDs (`content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id`) — pure identifiers, no predictive signal. Also excluded from the feature set (but used for filtering): `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — availability flags, not content signals.

In [22]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd

fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.


In [23]:
from datasets import load_dataset
ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [24]:
ds2 = load_dataset("FlyRank/internship-warehouse", "dim_content", streaming=True, split="train")

In [25]:
from huggingface_hub import list_repo_files
files = list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
fact_files = sorted(f for f in files if "fact_content_daily_performance" in f)
print(fact_files)

['fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet', 'fact_content_daily_performance/month=2025-07/data_0.parquet', 'fact_content_daily_performance/month=2025-08/data_0.parquet', 'fact_content_daily_performance/month=2025-09/data_0.parquet', 'fact_content_daily_performance/month=2025-10/data_0.parquet', 'fact_content_daily_performance/month=2025-11/data_0.parquet', 'fact_content_daily_performance/month=2025-12/data_0.parquet', 'fact_content_daily_performance/month=2026-01/data_0.parquet', 'fact_content_daily_performance/month=2026-02/data_0.parquet', 'fact_content_daily_performance/month=2026-03/data_0.parquet', 'fact_content_daily_performance/month=2026-04/data_0.p

In [26]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret "
    "(TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

sep_df = con.sql("""
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/'
        'fact_content_daily_performance/month=2025-09/data_0.parquet'
    )
""").df()

print(sep_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(845813, 31)


In [27]:
sample = list(ds.take(3))
sample[0]

{'report_date': datetime.date(2025, 1, 27),
 'client_hash_id': 'client_9958f0a7ae1df715',
 'content_hash_id': 'content_3b70a18ea133b2bb',
 'client_has_gsc': True,
 'client_has_ga4': True,
 'gsc_data_available': True,
 'ga4_data_available': False,
 'gsc_impressions': 30,
 'gsc_clicks': 0,
 'gsc_sum_position': 115,
 'gsc_avg_position': 3.8333333333333335,
 'ga4_pageviews': 0,
 'ga4_sessions': 0,
 'ga4_users': 0,
 'ga4_engaged_sessions': 0,
 'ga4_total_engagement_sec': 0,
 'sessions_organic': 0,
 'sessions_direct': 0,
 'sessions_referral': 0,
 'sessions_social': 0,
 'sessions_paid': 0,
 'sessions_ai': 0,
 'ai_chatgpt': 0,
 'ai_perplexity': 0,
 'ai_gemini': 0,
 'ai_copilot': 0,
 'ai_claude': 0,
 'ai_meta': 0,
 'ai_other': 0,
 'scroll_events': 0}

In [28]:
sample = list(ds2.take(3))
sample[0]

{'client_hash_id': 'client_04660893ae39614a',
 'content_hash_id': 'content_004de9653278b5a4',
 'keyword_hash_id': 'keyword_e754999ab88dd9f2',
 'url_hash_id': 'url_d6091f18cf628794',
 'keyword_char_count': 22,
 'keyword_token_count': 4,
 'url_char_count': 108,
 'content_created_date': datetime.date(2026, 5, 30),
 'content_updated_date': datetime.date(2026, 7, 1),
 'content_type': 'keyword article',
 'search_volume': 30,
 'competition': 0.91,
 'competition_level': 'HIGH',
 'cpc': 0.98,
 'main_intent': 'transactional',
 'backlinks': 16,
 'category_count': 3,
 'keyword_created_date': datetime.date(2026, 5, 12),
 'provider_used': 'gemini-generate-content',
 'model_used': 'gemini-3-flash-preview',
 'char_count': 15682,
 'word_count': 2555,
 'last_optimized_date': None,
 'optimization_eligible_date': None,
 'is_published': True,
 'is_deleted': False}

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


**Features — from `fact_content_daily_performance`, prior window (Sept 1–15) only, aggregated per `content_hash_id`**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`
- `gsc_days_with_data` (derived: `gsc_sum_position / gsc_avg_position`, rounded — recovers day-count so the raw sum becomes interpretable)
- `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai`
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other`
- `scroll_events`

**Features — from `dim_content` (static + point-in-time as of Sept 15)**
- `content_type`, `main_intent`, `provider_used`, `model_used`
- `search_volume`, `competition`, `cpc`, `backlinks`, `category_count`
- `word_count`, `keyword_char_count`, `keyword_token_count`, `url_char_count`
- `is_published`, `is_deleted`
- `content_age_days` (derived: Sept 15 − `content_created_date` — safe, set-once field)
- `keyword_age_days` (derived: Sept 15 − `keyword_created_date` — safe, set-once field)
- `is_optimization_eligible_yet` (derived: `optimization_eligible_date` ≤ Sept 15)

**Context (used for filtering/grouping, never as a feature)**
- `client_hash_id` — grouping key for the train/test split
- `report_date` — defines the prior/later windows, not itself a feature
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — flags for which rows are trustworthy

**Excluded**
- `content_hash_id`, `keyword_hash_id`, `url_hash_id` — pure identifiers, no predictive signal
- `gsc_impressions`, later window — this is the label source (two-gate rule, gate 1)
- `competition_level` — redundant tier derived from `competition`
- `char_count` — redundant with `word_count`
- `gsc_sum_position` — superseded by derived `gsc_days_with_data`
- `days_since_last_optimized` — 0% coverage after leakage-safe gating (14,812/53,127
  rows would leak by being optimized after the cutoff; the remaining 38,315 were never
  optimized at all as of this slice); not usable as a feature.
- `is_optimization_eligible_yet` — 100% of rows resolve to `False` as of the Sept 15
  cutoff (0/53,127 eligible on or before cutoff); constant value, zero variance, no
  predictive signal regardless of coverage. Also noted: identical counts to
  `last_optimized_date`'s gating check, suggesting the two columns may share an
  underlying source event.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 - Grain Check**

In [29]:
print("Total rows:", len(sep_df))
print("Unique (content_hash_id, report_date) pairs:", sep_df.drop_duplicates(['content_hash_id','report_date']).shape[0])

Total rows: 845813
Unique (content_hash_id, report_date) pairs: 845813


Confirms one row really is one content×day — same check as before, rerun on the new slice.

**Query 2 — Row count + date span**
September 2025, a full calendar month.

In [30]:
print("Row count:", len(sep_df))
print("Date range:", sep_df['report_date'].min(), "to", sep_df['report_date'].max())
print("Unique content items:", sep_df['content_hash_id'].nunique())
print("Unique clients:", sep_df['client_hash_id'].nunique())

Row count: 845813
Date range: 2025-09-01 00:00:00 to 2025-09-30 00:00:00
Unique content items: 53127
Unique clients: 23


Date range is exactly Sept 1 to Sept 30, no partition bleed at either edge.

**Query 3 - Availability, `IS_TRUE`**

In [31]:
available = sep_df[sep_df['gsc_data_available'] == True]
print("Rows with gsc_data_available == True:", len(available), "out of", len(sep_df))
print(f"That's {len(available)/len(sep_df):.1%} of the slice")

Rows with gsc_data_available == True: 845813 out of 845813
That's 100.0% of the slice


This result deserves suspicion, not acceptance. 100.0% availability — literally every single row has gsc_data_available == True, zero exceptions across 845,813 rows. That's an unusually clean result for a real-world flag. Two possibilities:

It's genuinely true for this table (maybe gsc_data_available is only ever False for clients who never opted into GSC at all, and those clients' rows simply don't exist in this table — i.e., the flag doesn't vary within this table, only across which clients get included).
The flag isn't doing what Section 2 assumes it does, and the real filtering already happened upstream before you ever got the data.

In [32]:
rel = "hf://datasets/FlyRank/internship-warehouse"

content_ids = tuple(sep_df['content_hash_id'].unique())

dim_df = con.sql(f"""
    SELECT * FROM read_parquet('{rel}/dim_content.parquet')
    WHERE content_hash_id IN {content_ids}
""").df()

dim_df.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(53127, 26)

In [33]:
import pandas as pd
cutoff = pd.Timestamp('2025-09-15')

# Ungated (naive) version — what you'd get if you didn't check the cutoff
dim_df['content_updated_date'] = pd.to_datetime(dim_df['content_updated_date'])
dim_df['days_since_update_naive'] = (cutoff - dim_df['content_updated_date']).dt.days

# How many rows would this leak on? (negative days = update happened AFTER the cutoff)
leaked = dim_df[dim_df['days_since_update_naive'] < 0]
print(f"Rows where naive calc goes negative (i.e. would leak future info): {len(leaked)} / {len(dim_df)}")

# Gated version — the fix
dim_df['days_since_update'] = dim_df.apply(
    lambda r: r['days_since_update_naive'] if r['content_updated_date'] <= cutoff else None,
    axis=1
)
print("Gated version, min value:", dim_df['days_since_update'].min())
print("Gated version, rows set to null (update was after cutoff):", dim_df['days_since_update'].isna().sum())

Rows where naive calc goes negative (i.e. would leak future info): 52855 / 53127
Gated version, min value: 7.0
Gated version, rows set to null (update was after cutoff): 52855


In [34]:
dim_df['last_optimized_date'] = pd.to_datetime(dim_df['last_optimized_date'])
dim_df['days_since_optimized_naive'] = (cutoff - dim_df['last_optimized_date']).dt.days

leaked_opt = dim_df[dim_df['days_since_optimized_naive'] < 0]
print(f"Rows where naive calc goes negative: {len(leaked_opt)} / {len(dim_df)}")

# also check how many are NaT (never optimized at all — different from "optimized late")
never_optimized = dim_df['last_optimized_date'].isna().sum()
print(f"Rows with no optimization date at all (NaT): {never_optimized} / {len(dim_df)}")

dim_df['days_since_last_optimized'] = dim_df.apply(
    lambda r: r['days_since_optimized_naive'] if pd.notna(r['last_optimized_date']) and r['last_optimized_date'] <= cutoff else None,
    axis=1
)
print("Gated version, rows with a valid value:", dim_df['days_since_last_optimized'].notna().sum())

Rows where naive calc goes negative: 14812 / 53127
Rows with no optimization date at all (NaT): 38315 / 53127
Gated version, rows with a valid value: 0


In [35]:
dim_df['optimization_eligible_date'] = pd.to_datetime(dim_df['optimization_eligible_date'])

eligible_before_cutoff = dim_df[dim_df['optimization_eligible_date'] <= cutoff]
eligible_after_cutoff = dim_df[dim_df['optimization_eligible_date'] > cutoff]
never_eligible = dim_df['optimization_eligible_date'].isna().sum()

print(f"Eligible on/before Sept 15: {len(eligible_before_cutoff)} / {len(dim_df)}")
print(f"Eligible after Sept 15 (i.e. False as of cutoff): {len(eligible_after_cutoff)} / {len(dim_df)}")
print(f"No eligible date at all (NaT): {never_eligible} / {len(dim_df)}")

Eligible on/before Sept 15: 0 / 53127
Eligible after Sept 15 (i.e. False as of cutoff): 14812 / 53127
No eligible date at all (NaT): 38315 / 53127


**Result**
- `content_updated_date` gating check:**
52,855 / 53,127 rows (99.5%) would leak future information under a naive (ungated)
`days_since_update` calculation — i.e. the update happened after the Sept 15 cutoff.
After gating, only 272 rows (0.5%) retain a valid value.


- `last_optimized_date` gating check:**
Of 53,127 rows: 14,812 (27.9%) would leak (optimized after the cutoff), and 38,315
(72.1%) have no optimization date at all (never optimized as of this slice). Together
these account for 100% of rows — after gating, **zero rows** have a valid
`days_since_last_optimized` value.

- `is_optimization_eligible_yet` gating check:**
0 / 53,127 rows are eligible on or before Sept 15. The 14,812 rows eligible after the
cutoff resolve to `False` (not yet eligible), and the 38,315 rows with no eligibility
date (NaT) also resolve to `False` (no eligibility set). Combined: **100% of rows are
`False` as of this cutoff** — zero variance, not missingness

**Conclusion:** Both derived date features are dropped. The gating logic itself is
proven correct — it successfully caught and nulled out every leaking value in both
cases — but the underlying columns don't support a usable feature at this cutoff:
nearly all content in this slice was either updated or optimized *after* September 15,
not before it. This suggests content updates/optimization in this dataset cluster
later in a content's lifecycle relative to any early-to-mid window, which is itself
worth noting as a data-limits observation in Section 4 rather than just a dropped
feature.

**Query 4 — Client concentration and date coverage**

In [36]:
import duckdb

client_concentration = duckdb.sql("""
    SELECT
        client_hash_id,
        COUNT(*) AS row_count,
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER () AS pct_of_rows,
        COUNT(DISTINCT content_hash_id) AS unique_content_items,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM sep_df
    GROUP BY client_hash_id
    ORDER BY row_count DESC
""").df()

print(client_concentration)

top3_share = client_concentration['pct_of_rows'].head(3).sum()
print(f"\nTop 3 clients combined share of rows: {top3_share:.1f}%")

# window sanity check — does every client's data span the full Sept 1–30 range?
incomplete_window = client_concentration[
    (client_concentration['first_date'] > pd.Timestamp('2025-09-01')) |
    (client_concentration['last_date'] < pd.Timestamp('2025-09-30'))
]
print(f"\nClients with incomplete date coverage: {len(incomplete_window)} / {len(client_concentration)}")

             client_hash_id  row_count  pct_of_rows  unique_content_items  \
0   client_73cda7b4e4f265ea     342437    40.486136                 16672   
1   client_62f4a7e64f5e0096     213481    25.239740                 11236   
2   client_9958f0a7ae1df715      89733    10.609083                  3067   
3   client_fef1a8f436438636      70485     8.333402                  3604   
4   client_3197e6291363b4db      30077     3.555987                  3216   
5   client_23a62021009f63c4      24347     2.878532                  3520   
6   client_65de48885f4ef01b      15919     1.882095                  1284   
7   client_ff644d8251367cbb      15731     1.859867                  1656   
8   client_08a6a72ff48e62c0      14764     1.745539                  4413   
9   client_b10cb2997d0c7c86      14219     1.681104                  1822   
10  client_c182d11e4862a37d       4645     0.549176                   251   
11  client_d211cb07b9059bab       2547     0.301130                   637   

Severe concentration: top 3 clients account for 76.3% of rows (client `73cda7b4`
alone is 40.5%). Any grouped train/test split must account for this explicitly —
a naive random group split risks putting the dominant client entirely in one fold.

Date-coverage check surfaced a second issue: 8 of 23 clients have zero rows before
Sept 24 (7 starting exactly on Sept 24, one on Sept 27). These clients have no
presence at all in the Sept 1–15 prior window used for feature construction.

**Query 5 — Late-cohort history check (August 2025 partition)**

In [37]:
late_clients = (
    'client_23a62021009f63c4', 'client_08a6a72ff48e62c0', 'client_795153d5b7850ccf',
    'client_4a18d1793d92fb84', 'client_0e1acc6cd57b0eba', 'client_2910fd937f0b4d9a',
    'client_59256b0571e0c970', 'client_599043c0ff13edea'
)

rel = "hf://datasets/FlyRank/internship-warehouse"

aug_check = con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2025-08/data_0.parquet')
    WHERE client_hash_id IN {late_clients}
    GROUP BY client_hash_id
""").df()

print(aug_check)
print(f"\nOf the 8 late clients, {len(aug_check)} appear in August at all.")

Empty DataFrame
Columns: [client_hash_id, row_count, first_date, last_date]
Index: []

Of the 8 late clients, 0 appear in August at all.


Zero of the 8 late-arriving clients appear in the August 2025 partition either —
rules out a simple September-only data-pull gap.

**Query 6 — Late-cohort content creation dates**

In [38]:
late_clients = (
    'client_23a62021009f63c4', 'client_08a6a72ff48e62c0', 'client_795153d5b7850ccf',
    'client_4a18d1793d92fb84', 'client_0e1acc6cd57b0eba', 'client_2910fd937f0b4d9a',
    'client_59256b0571e0c970', 'client_599043c0ff13edea'
)

# content items belonging to these 8 clients, from the fact table
late_content_ids = sep_df[sep_df['client_hash_id'].isin(late_clients)]['content_hash_id'].unique()
print(f"Unique content items for these 8 clients: {len(late_content_ids)}")

# their creation dates, from dim_content
late_dim = dim_df[dim_df['content_hash_id'].isin(late_content_ids)].copy()
late_dim['content_created_date'] = pd.to_datetime(late_dim['content_created_date'])

print(late_dim['content_created_date'].describe())
print("\nCreation date value counts (by day):")
print(late_dim['content_created_date'].dt.date.value_counts().sort_index())

# compare: how does this distribution compare to the OTHER 15 clients' content?
other_content_ids = sep_df[~sep_df['client_hash_id'].isin(late_clients)]['content_hash_id'].unique()
other_dim = dim_df[dim_df['content_hash_id'].isin(other_content_ids)].copy()
other_dim['content_created_date'] = pd.to_datetime(other_dim['content_created_date'])
print("\nFor comparison, other clients' creation dates — earliest and latest:")
print(other_dim['content_created_date'].min(), "to", other_dim['content_created_date'].max())

Unique content items for these 8 clients: 8525
count                          8525
mean     2025-07-31 21:43:41.137830
min             2025-04-15 00:00:00
25%             2025-07-14 00:00:00
50%             2025-08-13 00:00:00
75%             2025-08-27 00:00:00
max             2025-09-21 00:00:00
Name: content_created_date, dtype: object

Creation date value counts (by day):
content_created_date
2025-04-15    182
2025-04-16     44
2025-04-22    327
2025-05-07    411
2025-05-14     54
             ... 
2025-09-08      2
2025-09-17     38
2025-09-18    887
2025-09-19    476
2025-09-21    101
Name: count, Length: 70, dtype: int64

For comparison, other clients' creation dates — earliest and latest:
2025-01-08 00:00:00 to 2026-03-06 00:00:00


Content for these 8 clients' items substantially predates their Sept 24 fact-table
appearance: median `content_created_date` Aug 13, 2025, earliest Apr 15, 2025 — vs.
a Sept 24–27 first appearance in the fact table. This rules out "genuinely new
client" as the explanation; the content existed for weeks to months before tracking
began.

**Query 7 — Batch-activation check (daily active-client trend)**

In [40]:
daily_summary = duckdb.sql("""
    SELECT
        report_date,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS active_clients,
        COUNT(DISTINCT content_hash_id) AS active_content_items
    FROM sep_df
    GROUP BY report_date
    ORDER BY report_date
""").df()

print(daily_summary.to_string())

   report_date  total_rows  active_clients  active_content_items
0   2025-09-01       25462              15                 25462
1   2025-09-02       25144              15                 25144
2   2025-09-03       24878              15                 24878
3   2025-09-04       26426              15                 26426
4   2025-09-05       26579              15                 26579
5   2025-09-06       26872              15                 26872
6   2025-09-07       27804              15                 27804
7   2025-09-08       28239              15                 28239
8   2025-09-09       28544              15                 28544
9   2025-09-10       28078              15                 28078
10  2025-09-11       27742              15                 27742
11  2025-09-12       26494              15                 26494
12  2025-09-13       24471              15                 24471
13  2025-09-14       24921              15                 24921
14  2025-09-15       2525

Daily active-client count holds flat at 15 for Sept 1–23, then jumps to 22 on
Sept 24 — a discrete step, not a ramp. The original 15 clients show no volume
disruption on or around that date, ruling out a platform-wide event. This is a
synchronized batch activation isolated to the 8-client cohort. Post-activation
reporting is also intermittent (19–22 active clients/day through month-end) —
cause of the activation itself (system migration, contract start, integration
going live) is unconfirmed; only the timing and isolation are directly observed.

In [41]:
prior_window = sep_df[
    (sep_df['report_date'] >= '2025-09-01') & (sep_df['report_date'] <= '2025-09-15')
]

fact_feature_cols = [
    'gsc_impressions', 'gsc_clicks', 'gsc_avg_position',
    'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec',
    'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai',
    'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other',
    'scroll_events'
]
print("--- Fact table feature null rates (prior window only) ---")
print(prior_window[fact_feature_cols].isna().mean().sort_values(ascending=False))

dim_feature_cols = [
    'content_type', 'main_intent', 'provider_used', 'model_used',
    'search_volume', 'competition', 'cpc', 'backlinks', 'category_count',
    'word_count', 'keyword_char_count', 'keyword_token_count', 'url_char_count',
    'is_published', 'is_deleted'
]
print("\n--- dim_content feature null rates ---")
print(dim_df[dim_feature_cols].isna().mean().sort_values(ascending=False))

--- Fact table feature null rates (prior window only) ---
gsc_impressions             0.0
gsc_clicks                  0.0
gsc_avg_position            0.0
ga4_pageviews               0.0
ga4_sessions                0.0
ga4_users                   0.0
ga4_engaged_sessions        0.0
ga4_total_engagement_sec    0.0
sessions_organic            0.0
sessions_direct             0.0
sessions_referral           0.0
sessions_social             0.0
sessions_paid               0.0
sessions_ai                 0.0
ai_chatgpt                  0.0
ai_perplexity               0.0
ai_gemini                   0.0
ai_copilot                  0.0
ai_claude                   0.0
ai_meta                     0.0
ai_other                    0.0
scroll_events               0.0
dtype: float64

--- dim_content feature null rates ---
provider_used          0.999962
backlinks              0.737177
word_count             0.498315
model_used             0.429330
cpc                    0.092401
competition            

In [42]:
print("--- word_count missingness by content_type ---")
print(dim_df.groupby('content_type')['word_count'].apply(lambda x: x.isna().mean()))

print("\n--- backlinks missingness by content_type ---")
print(dim_df.groupby('content_type')['backlinks'].apply(lambda x: x.isna().mean()))

print("\n--- search_volume/competition/cpc: do the same 9.24% rows overlap exactly? ---")
same_rows = (dim_df['search_volume'].isna() == dim_df['competition'].isna()) & \
            (dim_df['competition'].isna() == dim_df['cpc'].isna())
print(f"Rows where all three agree on missing/present: {same_rows.mean():.4%}")

print("\n--- provider_used: how many rows even have a non-null value? ---")
print(dim_df['provider_used'].value_counts(dropna=False))

--- word_count missingness by content_type ---
content_type
comparison article    0.000000
feedly article        0.019526
keyword article       0.538345
Name: word_count, dtype: float64

--- backlinks missingness by content_type ---
content_type
comparison article    0.500000
feedly article        1.000000
keyword article       0.715224
Name: backlinks, dtype: float64

--- search_volume/competition/cpc: do the same 9.24% rows overlap exactly? ---
Rows where all three agree on missing/present: 100.0000%

--- provider_used: how many rows even have a non-null value? ---
provider_used
None      53125
openai        2
Name: count, dtype: int64


In [43]:
print(dim_df.groupby('content_type')['search_volume'].apply(lambda x: x.isna().mean()))

content_type
comparison article    0.000000
feedly article        1.000000
keyword article       0.016562
Name: search_volume, dtype: float64


## Missing values

*Which fields go blank, how often, and is it random or does it follow a pattern?*

**Fact table (daily metrics): 0% missing, all 22 feature columns.** Checked across
the prior window (Sept 1–15). Every row has a value for every metric — likely because
these are counts that get filled with 0 when nothing happened, not left blank.

**`dim_content` fields: several are missing, but the pattern is not random — it
follows `content_type`.**

- `provider_used` — 99.996% missing (only 2 rows out of 53,127 have a value). Too
  empty to use. Moved to Excluded.

- `word_count` — missing 0% of the time for comparison articles, 2% for feedly
  articles, but 53.8% for keyword articles. The gap is explained by content type,
  not random.

- `backlinks` — missing 50% (comparison articles), 100% (feedly articles — never
  tracked), 71.5% (keyword articles). Same story: explained by content type.

- `search_volume`, `competition`, `cpc` — all three go missing together on the
  exact same rows (confirmed: 100% overlap). Checked against content type: 0%
  missing for comparison articles, 100% missing for feedly articles, 1.7% missing
  for keyword articles. Also explained by content type.

**Decision: keep these fields as features, missing values stay missing.** No extra
"is missing" flag needed, since `content_type` already explains the pattern —
adding one would just repeat information the model already has. Tree-based models
can handle missing values natively; if you use logistic regression later, you'll
need an explicit imputation step at that point.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **No pre-built label exists.** Unlike the old starter CSV's `trend_direction`, this table
  has no ground-truth trend label. `trend_direction` here is a proxy constructed from
  `gsc_impressions` (prior 15 days vs. later 15 days, % change, bucketed). It reflects a
  modeling choice, not a verified outcome — a different metric (e.g. `gsc_clicks`) or a
  different split ratio could produce a different label for the same content.

- **Single-month slice, not the full panel.** The full dataset spans 18 months
  (`2025-01` through `2026-06`, confirmed via the month-partitioned file listing).
  September 2025 was chosen as a genuine mid-panel month, clear of both the panel's
  start (no prior history) and end (no future window). But findings from this one month
  — availability rates, date-field behavior, client counts — are not guaranteed to hold
  in other months, especially given likely seasonal effects on search/content behavior.

- **Availability flags showed zero variance in this slice.** `gsc_data_available` and
  `client_has_gsc` were both `True` for all 845,813 rows — confirmed via cross-tab. This
  means the September slice happens to contain no non-GSC clients; the filtering already
  happened upstream, not row-by-row within this table. A different month or the full
  panel could show real variation, so this can't be assumed to generalize.

- **Content lifecycle fields cluster heavily after the decision cutoff.** Three derived
  date-based features (`days_since_update`, `days_since_last_optimized`,
  `is_optimization_eligible_yet`) were tested with leakage-safe gating at the Sept 15
  cutoff and all failed: 99.5% of update events and 100% of optimization events occurred
  *after* the cutoff (or never occurred at all) in this slice. This means the dataset
  can't currently answer "how fresh was this content as of the decision point" — a gap,
  not just a dropped feature.

- **Observed field redundancy, not investigated further.** `last_optimized_date` and
  `optimization_eligible_date` produced identical leak/never-occurred counts (14,812 /
  38,315), suggesting they may derive from the same underlying event rather than being
  independent. Flagged, not confirmed.

- **Client concentration not yet checked for this slice.** The old CSV showed 2 of 32
  clients holding ~33% of rows combined, motivating the grouped-split requirement. This
  slice has 23 unique clients — whether a similar concentration exists here is unverified
  and should be checked before the train/test split is built.

- **Anonymization artifact, unresolved.** An earlier `dim_content` sample showed a
  `content_created_date` of 2026-05-30 — later than any date in.

- **Client concentration is severe.** Top 3 of 23 clients account for 76.3% of rows
  (client `73cda7b4` alone: 40.5%). This is more concentrated than the old starter
  CSV's 2-of-32-clients-at-33% finding that originally motivated the grouped-split
  requirement. Any `GroupKFold`/`StratifiedGroupKFold` by `client_hash_id` needs to
  account for this explicitly, since a naive random group assignment could put the
  dominant client entirely in train or entirely in test.

- **A cohort of 8 clients was batch-activated into tracking on/around Sept 24, 2025.**
  Daily active-client counts are flat at 15 for Sept 1–23, then jump to 22 with no
  corresponding disruption to the original 15 clients' volume — ruling out a
  platform-wide event. Associated content for these clients predates activation by
  weeks to months (median `content_created_date` Aug 13, earliest Apr 15), so this
  reflects a tracking-pipeline event, not new client content. These 8 clients have
  zero rows in the Sept 1–15 prior window and are excluded from this slice's
  feature/label construction. Cause of the activation is unconfirmed.

## 5. Output

*What does this analysis hand to the human, in one sentence?*

A ranked list of content items — for those with sufficient prior-window history —
flagged by predicted trend direction (up/down/stable), ordered for human review
and evaluated primarily by Precision@K; a decision-support signal, not an
automated action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.